In [62]:
import pandas as pd

In [63]:
data = pd.read_csv('DateFruit_Dataset.csv')
data.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [64]:
X = data.drop('Class',axis = 1)
y = data['Class']

In [65]:
y.unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [66]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y=le.fit_transform(y)

In [67]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.3,random_state = 42)

In [68]:
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
X_train_scaled = ss.fit_transform(X_train)
X_test_scaled = ss.transform(X_test)

In [69]:
import torch 
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled,dtype = torch.float32)
X_test_tensor = torch.tensor(X_test_scaled,dtype = torch.float32)

y_train_tensor = torch.tensor(y_train,dtype=torch.long)
y_test_tensor = torch.tensor(y_test,dtype=torch.long)

In [70]:
from torch.utils.data import TensorDataset,DataLoader

train_tensor = TensorDataset(X_train_tensor,y_train_tesnsor)
test_tensor = TensorDataset(X_test_tensor,y_test_tesnsor)

train_loader = DataLoader(train_tensor, batch_size=64,shuffle = True)
test_loader = DataLoader(test_tensor, batch_size=64)

## ANN

In [71]:
class ANNclassifier(nn.Module):
    def __init__(self):
        super(ANNclassifier,self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X_train.shape[1],8),
            nn.ReLU(),

            nn.Linear(8,8),
            nn.ReLU(),

            nn.Linear(8,7),
        )

    def forward(self,x):
        return self.model(x)

In [72]:
#loss and optimizer
import torch.optim as optim
model = ANNclassifier()
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [80]:
epochs = 100
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for xb,yb in train_loader:
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criteria(outputs,yb)
        loss.backward()
        optimizer.step() # updates parameters
        running_loss += loss.item()

    train_losses = running_loss/len(train_loader)

    print(f"epoch = {epoch+1}/{epochs} ==> loss = {train_losses}")

epoch = 1/100 ==> loss = 0.09888568297028541
epoch = 2/100 ==> loss = 0.09894021414220333
epoch = 3/100 ==> loss = 0.09933399930596351
epoch = 4/100 ==> loss = 0.09665874689817429
epoch = 5/100 ==> loss = 0.09741947650909424
epoch = 6/100 ==> loss = 0.09784510284662247
epoch = 7/100 ==> loss = 0.0986614715307951
epoch = 8/100 ==> loss = 0.09556625485420227
epoch = 9/100 ==> loss = 0.09768591672182084
epoch = 10/100 ==> loss = 0.09744539074599742
epoch = 11/100 ==> loss = 0.09646724238991737
epoch = 12/100 ==> loss = 0.09570591673254966
epoch = 13/100 ==> loss = 0.09514252096414566
epoch = 14/100 ==> loss = 0.09635305255651475
epoch = 15/100 ==> loss = 0.09468048699200153
epoch = 16/100 ==> loss = 0.09533598572015763
epoch = 17/100 ==> loss = 0.09434546176344157
epoch = 18/100 ==> loss = 0.09480388462543488
epoch = 19/100 ==> loss = 0.09500758275389672
epoch = 20/100 ==> loss = 0.09468371123075485
epoch = 21/100 ==> loss = 0.09563852697610856
epoch = 22/100 ==> loss = 0.0965512886643409

In [81]:
model.eval()
total = 0
correct = 0

with torch.no_grad():
    for xb,yb in test_loader:
        outputs = model(xb)
        _,predicted = torch.max(outputs,1)

        correct += (predicted==yb).sum().item()
        total += yb.size(0)

print('total vals: ',total)
print("correct vals: ",correct)

total vals:  270
correct vals:  243
